In [0]:
from pyspark.sql.functions import col, explode, explode_outer, from_json, schema_of_json, get_json_object
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Example 1: Read nested JSON from a file
# Replace with your actual file path (e.g., "/Volumes/catalog/schema/volume/data.json")
# df = spark.read.json("/path/to/your/nested.json")

# Example 2: Create sample nested JSON data for demonstration
sample_data = [
    (1, '{"name": "John", "age": 30, "address": {"city": "NYC", "zip": "10001"}, "hobbies": ["reading", "gaming"]}'),
    (2, '{"name": "Jane", "age": 25, "address": {"city": "LA", "zip": "90001"}, "hobbies": ["sports", "music", "travel"]}'),
    (3, '{"name": "Bob", "age": 35, "address": {"city": "Chicago", "zip": "60601"}, "hobbies": ["cooking"]}')
]

df_raw = spark.createDataFrame(sample_data, ["id", "json_string"])

# Infer schema from JSON string
json_schema = schema_of_json(df_raw.select("json_string").first()[0])
print(f"Inferred Schema: {json_schema}")

# Parse JSON string into structured columns
df = df_raw.withColumn("data", from_json(col("json_string"), json_schema))

print("\n=== Original Data with Parsed JSON ===")
display(df.select("id", "data"))

In [0]:
# Access nested fields using dot notation or col()
df_flattened = df.select(
    col("id"),
    col("data.name").alias("name"),
    col("data.age").alias("age"),
    col("data.address.city").alias("city"),
    col("data.address.zip").alias("zip"),
    col("data.hobbies").alias("hobbies")
)

print("\n=== Flattened Data (nested struct accessed) ===")
display(df_flattened)

In [0]:
# Explode array to create one row per hobby
df_exploded = df_flattened.select(
    col("id"),
    col("name"),
    col("age"),
    col("city"),
    explode(col("hobbies")).alias("hobby")
)

print("\n=== Exploded Array Data (one row per hobby) ===")
display(df_exploded)

In [0]:
# If your data might have null arrays, use explode_outer to preserve rows
# explode_outer keeps the row even if the array is null or empty
df_exploded_outer = df_flattened.select(
    col("id"),
    col("name"),
    explode_outer(col("hobbies")).alias("hobby")
)

print("\n=== Using explode_outer (preserves nulls) ===")
display(df_exploded_outer)

In [0]:
# Alternative: Extract fields directly from JSON string using get_json_object
df_extracted = df_raw.select(
    col("id"),
    get_json_object(col("json_string"), "$.name").alias("name"),
    get_json_object(col("json_string"), "$.age").alias("age"),
    get_json_object(col("json_string"), "$.address.city").alias("city"),
    get_json_object(col("json_string"), "$.hobbies[0]").alias("first_hobby")
)

print("\n=== Extracted using get_json_object ===")
display(df_extracted)

In [0]:
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import ArrayType, StringType, MapType
import json

# UDF to parse JSON string and extract a field
@udf(returnType=StringType())
def extract_json_field(json_str, field_name):
    """Extract a specific field from JSON string"""
    if json_str:
        try:
            data = json.loads(json_str)
            return str(data.get(field_name, None))
        except:
            return None
    return None

# UDF to extract nested field from JSON
@udf(returnType=StringType())
def extract_nested_json(json_str, *fields):
    """Extract nested field from JSON string (e.g., address.city)"""
    if json_str:
        try:
            data = json.loads(json_str)
            for field in fields:
                if data and isinstance(data, dict):
                    data = data.get(field)
                else:
                    return None
            return str(data) if data else None
        except:
            return None
    return None

# UDF to extract array from JSON
@udf(returnType=ArrayType(StringType()))
def extract_json_array(json_str, field_name):
    """Extract array field from JSON string"""
    if json_str:
        try:
            data = json.loads(json_str)
            arr = data.get(field_name, [])
            return arr if isinstance(arr, list) else []
        except:
            return []
    return []

# Test UDFs on the sample data
df_udf = df_raw.select(
    col("id"),
    extract_json_field(col("json_string"), lit("name")).alias("name_udf"),
    extract_nested_json(col("json_string"), lit("address"), lit("city")).alias("city_udf"),
    extract_json_array(col("json_string"), lit("hobbies")).alias("hobbies_udf")
)

print("\n=== Using Python UDF for JSON Processing ===")
display(df_udf)

# Now explode the array extracted by UDF
df_udf_exploded = df_udf.select(
    col("id"),
    col("name_udf"),
    col("city_udf"),
    explode(col("hobbies_udf")).alias("hobby")
)

print("\n=== UDF + Explode ===")
display(df_udf_exploded)

print("\n⚠️ Note: Built-in functions (from_json, explode) are faster than UDFs!")
print("UDFs require serialization overhead. Use built-in functions when possible.")

In [0]:
import json

# Regular Python function (not a UDF)
def parse_and_explode_json(json_string):
    """Regular Python function to parse JSON and explode arrays"""
    try:
        data = json.loads(json_string)
        name = data.get('name')
        age = data.get('age')
        city = data.get('address', {}).get('city')
        hobbies = data.get('hobbies', [])
        
        # Create one row per hobby (explode logic)
        results = []
        for hobby in hobbies:
            results.append({
                'name': name,
                'age': age,
                'city': city,
                'hobby': hobby
            })
        return results
    except:
        return []

# Test on a single JSON string (local Python, not Spark)
sample_json = '{"name": "John", "age": 30, "address": {"city": "NYC", "zip": "10001"}, "hobbies": ["reading", "gaming"]}'
result = parse_and_explode_json(sample_json)

print("\n=== Regular Python Function Result (local) ===")
for row in result:
    print(row)

print("\n" + "="*60)
print("Key Differences:")
print("="*60)
print("1. Regular Python Function:")
print("   - Works on LOCAL data (single machine)")
print("   - Cannot be applied directly to Spark DataFrame")
print("   - Fast for small data, doesn't scale")
print("\n2. UDF (User Defined Function):")
print("   - Wraps Python function for DISTRIBUTED execution")
print("   - Works on Spark DataFrame (across cluster)")
print("   - Scales to big data but has serialization overhead")
print("\n3. Built-in Spark Functions (from_json, explode):")
print("   - Optimized native Spark operations")
print("   - Fastest option, fully optimized by Catalyst")
print("   - Always prefer these over UDFs when possible")
print("="*60)

In [0]:
import pandas as pd

# Convert small Spark DataFrame to Pandas (local)
df_pandas = df_raw.limit(10).toPandas()  # Only for small data!

print("\n=== Original Pandas DataFrame ===")
print(df_pandas)

# Apply regular Python function to each row using pandas .apply()
exploded_rows = []
for idx, row in df_pandas.iterrows():
    parsed = parse_and_explode_json(row['json_string'])
    for item in parsed:
        item['id'] = row['id']  # Keep the ID
        exploded_rows.append(item)

df_result = pd.DataFrame(exploded_rows)

print("\n=== After Exploding with Regular Python Function ===")
print(df_result)

print("\n" + "="*60)
print("When to use each approach:")
print("="*60)
print("✅ Built-in Spark functions (from_json + explode):")
print("   → ANY size data, best performance")
print("\n✅ Regular Python + Pandas:")
print("   → Small data (<1M rows) that fits in memory")
print("   → Quick prototyping and analysis")
print("\n✅ UDF:")
print("   → Large distributed data")
print("   → Custom logic not available in built-in functions")
print("   → Slower than built-ins due to serialization")
print("="*60)

In [0]:
from pyspark.sql.functions import explode_outer, col
from pyspark.sql.types import ArrayType, StructType

def explode_all_arrays(df):
    """Recursively explode all arrays and flatten all structs in a DataFrame"""
    while True:
        found_array = False

        for field in df.schema.fields:
            if isinstance(field.dataType, ArrayType):
                df = df.withColumn(field.name, explode_outer(col(field.name)))
                found_array = True
                break

        if not found_array:
            break

        # Flatten struct columns
        struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]

        while struct_cols:
            struct_col = struct_cols.pop(0)

            expanded = [
                col(f"{struct_col}.{c.name}").alias(f"{struct_col}_{c.name}")
                for c in df.schema[struct_col].dataType.fields
            ]

            other_cols = [col(c) for c in df.columns if c != struct_col]

            df = df.select(*other_cols, *expanded)

            struct_cols = [
                f.name for f in df.schema.fields
                if isinstance(f.dataType, StructType)
            ]

    return df

print("\n=== Original DataFrame (from Cell 2 - already has some flattening) ===")
print(f"Schema: {df_flattened.schema}")
print(f"Row count: {df_flattened.count()}")
display(df_flattened)

# Apply the recursive explode function
print("\nApplying explode_all_arrays...")
df_fully_flattened = explode_all_arrays(df_flattened)

print("\n=== Fully Flattened DataFrame (all arrays exploded, all structs flattened) ===")
print(f"Schema: {df_fully_flattened.schema}")
print(f"Row count: {df_fully_flattened.count()}")
display(df_fully_flattened)